# CS781 Capstone · Phase 1
## Four-source corpus and 16K byte-BPE preparation

All four Phase 1 model architectures use the same `prep_002_mixed16k` preparation.
This notebook follows the preparation from document ownership through tokenizer fitting,
cache construction, and prompt-length analysis.

| Component | Experimental setup |
|---|---|
| Corpus, in ownership order | Supreme Court, High Courts, Acts, Tribunals |
| Document split | Hold out a document if the first 16 hex digits of its full-text SHA-256, interpreted as an integer, are divisible by 20 |
| Exact deduplication | Keep the first occurrence of each full-text hash in source and file order |
| Tokenizer | Byte BPE fitted on 500,000 training characters per source, with 16,000 IDs including the reserved IDs |
| Training caches | Up to 75M tokens per source in little-endian uint16 arrays, with separate held-out caches |
| Generation validation | 100 source-balanced cases and 50 Supreme Court anchors, with 25 shared cases |
| Model input setup | Equal source mixture, 2,560-token training blocks, 512-word generation prompts and 128-word references |


## 1. Run configuration

| Mode | Inputs | Work performed |
|---|---|---|
| `reuse` | Attach the extracted `prep_002_mixed16k` folder from `prep-2` | Read the saved document audit, tokenizer fit and token caches. Reassemble the validation groups, check the saved artifacts, and tokenize the prompts again to measure their lengths. |
| `rebuild` | Access to the pinned official corpus on Hugging Face | Stream and split the raw documents, recover the 50 anchors, fit byte BPE, encode the caches, and produce a complete preparation. Matching checkpoints in the working directory can be resumed. |

Use a **CPU** session and run the cells in order. For `reuse`, leave the default settings.
The attached folder should contain `preparation_config.json`, `cache_manifest.json`,
`tokenizer_state.pkl`, `audits/`, `caches/`, and the saved validation and test JSON files.
Set `PREP_INPUT` to the inner folder if it is renamed or detection finds several copies.

For `rebuild`, enable Internet and add the Kaggle secret `HF` if corpus access requires it.
Set `PREP_INPUT` only to restore a partial preparation. Optionally set `ANCHOR_INPUT` to an
earlier `prep_001` folder containing its configuration, sample manifest, prompts, and references.
With the default settings the 50 anchors are recovered directly from the pinned Supreme Court
file. Allow for a multi-hour CPU run and enough disk for the largest raw file plus the caches.
The saved tokenizer fit alone took about 28 minutes.

Locally, use Python 3 and Jupyter with `numpy` and `pandas` installed, set the input paths
explicitly, and use the `HF_TOKEN` environment variable if authentication is needed.
Rebuild mode also installs `ijson` and `huggingface-hub`.


In [1]:
from pathlib import Path

PREP_MODE = "reuse"                    # "reuse" or "rebuild"
PREP_INPUT = None                       # Extracted prep_002_mixed16k folder
ANCHOR_INPUT = None                     # Optional prep_001 folder for rebuild
RAW_INPUT_ROOT = None                   # Optional directory with pinned raw training files
HF_SECRET_NAME = "HF"
VERIFY_CACHE_HASHES = True
CREATE_PREPARATION_ZIP = False

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_ROOT = WORK_ROOT / "phase1_preparation_outputs"
PREPARATION_ID = "prep_002_mixed16k"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
DATASET_REVISION = "89edba6a44d6bb9593fd13b6895b5cd3b06eb82e"
SOURCE_ORDER = ["supreme_court", "high_courts", "acts", "tribunals"]
VOCAB_SIZE = 16_000
MAX_SEQ_LEN = 2560
BPE_CHARS_PER_SOURCE = 500_000
VALIDATION_CHARS_PER_SOURCE = 500_000
CACHE_TOKENS_PER_SOURCE = 75_000_000

if PREP_MODE not in {"reuse", "rebuild"}:
    raise ValueError('Set PREP_MODE to "reuse" or "rebuild".')
REBUILD = PREP_MODE == "rebuild"
print(f"Preparation mode · {PREP_MODE}")


Preparation mode · reuse


## 2. Project source and dependencies

The notebook embeds the original tokenizer and corpus processing modules. Raw corpus JSON is
streamed a record at a time with `ijson` in rebuild mode. Reuse mode reads the mounted preparation
directly and writes only diagnostics to the working directory.


In [2]:
from pathlib import Path
from importlib.metadata import version
import gc
import json
import os
import pickle
import platform
import shutil
import sys
from tempfile import TemporaryDirectory
import time

import pandas as pd
from IPython.display import FileLink, display

import re
from functools import lru_cache
import numpy as np
from IPython import get_ipython

if REBUILD:
    print("Installing raw-corpus dependencies", flush=True)
    get_ipython().run_line_magic("pip", "install -q ijson huggingface-hub")


In [3]:
# Project source carried by this notebook for fresh sessions.
modules = {
    'phase1_utils.py': r'''"""Small file and JSON helpers used by the Phase-1 notebooks."""
import hashlib
import json
import os
import pickle
import zipfile
from pathlib import Path


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def atomic_pickle(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "wb") as file:
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temporary, path)


def checked_json(path, value):
    path = Path(path)
    if path.exists() and read_json(path) != value:
        raise ValueError(f"Saved settings differ: {path}. Use a new experiment ID.")
    atomic_json(path, value)


def write_source(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    raw = content.encode("utf-8") if isinstance(content, str) else content
    if path.exists():
        if path.read_bytes() != raw:
            raise ValueError(f"Saved source differs: {path}. Keep its code or use a new experiment ID.")
        return
    path.write_bytes(raw)


def make_bundle(folder, output_path, archive_root):
    folder, output_path, archive_root = Path(folder), Path(output_path), Path(archive_root)
    temporary = output_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts and not path.name.endswith(".tmp"):
                archive.write(path, path.relative_to(archive_root))
    os.replace(temporary, output_path)
    return output_path
''',
    'phase1_paths.py': r'''"""Restore extracted Kaggle input folders, without assuming the ZIP survives."""
import shutil
import zipfile
from pathlib import Path
from phase1_utils import sha256_file


def find_input_folder(folder_name, marker, explicit=None, identical_files=()):
    if explicit:
        folder = Path(explicit)
        if not (folder / marker).is_file():
            raise FileNotFoundError(f"Expected {marker} in {folder}")
        return folder
    matches = sorted({p.parent for p in Path("/kaggle/input").rglob(marker)
                      if p.parent.name == folder_name})
    if not matches:
        return None
    if len(matches) == 1:
        return matches[0]
    if identical_files and all(all((p / name).exists() for name in identical_files) for p in matches):
        fingerprints = [tuple(sha256_file(p / name) for name in identical_files) for p in matches]
        if len(set(fingerprints)) == 1:
            return matches[0]
    raise ValueError(f"Several input copies found. Set the explicit folder path: {matches}")


def restore_folder(destination, folder_name, marker, explicit=None):
    destination = Path(destination)
    if (destination / marker).exists():
        return destination
    source = find_input_folder(folder_name, marker, explicit)
    if source is not None:
        destination.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("Restored extracted input folder:", source, flush=True)
    return destination


def restore_zip(zip_path, working_root):
    """Optional path for a ZIP that Kaggle did not extract automatically."""
    working_root = Path(working_root).resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            destination = (working_root / member.filename).resolve()
            if not destination.is_relative_to(working_root):
                raise ValueError("Unsafe archive path")
            if destination.exists() and not member.is_dir():
                raise ValueError(f"ZIP would overwrite {destination}; restore into a fresh working folder")
        archive.extractall(working_root)
''',
    'a1_tokenizer.py': r'''"""The assignment byte-BPE algorithm, with checkpoints between merges.

Pair counting, lexicographic tie breaking, non-overlapping replacements and
merge-order encoding are unchanged from the completed Phase-1/A3 tokenizer.
"""
import pickle
import re
import time
from collections import Counter
from pathlib import Path
from phase1_utils import atomic_pickle, sha256_file, text_hash


class A1BPETokenizer:
    def __init__(self):
        specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.vocab = [s.encode("utf-8") for s in specials]
        self.vocab += [bytes([b]) for b in range(256)]
        self.boundary_id = len(self.vocab)
        self.vocab.append(b"")
        self.names = [s.encode("utf-8") for s in specials]
        self.names += [bytes([b]) for b in range(256)]
        self.names.append(b"</w>")
        self.merges = {}
        self.vocab_size = len(self.vocab)

    def fit(self, text, vocab_size, checkpoint_path=None, checkpoint_merges=100,
            stop_after_merges=None):
        if vocab_size < 261:
            raise ValueError("Vocabulary must accommodate bytes and reserved tokens")
        signature = {"text_sha256": text_hash(text), "vocab_size": vocab_size,
                     "algorithm_source_sha256": sha256_file(__file__)}
        checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        elapsed_before = 0.0
        if checkpoint_path is not None and checkpoint_path.exists():
            with open(checkpoint_path, "rb") as file:
                saved = pickle.load(file)
            if saved["signature"] != signature:
                raise ValueError("BPE checkpoint settings or source differ")
            self.__dict__.update(saved["tokenizer_state"])
            words = saved["words"]
            elapsed_before = saved["elapsed_seconds"]
            print("Resuming BPE vocabulary:", self.vocab_size, flush=True)
        else:
            if self.merges:
                raise ValueError("Use a fresh tokenizer or its saved fitting checkpoint")
            words = Counter(tuple(b + 4 for b in piece.encode("utf-8")) + (self.boundary_id,)
                            for piece in re.findall(r"\S+|\s+", text))
        started = time.monotonic()
        last_saved = started
        merges_this_call = 0
        commit_in_progress = False

        def snapshot():
            if checkpoint_path is not None:
                atomic_pickle(checkpoint_path, {"signature": signature,
                    "tokenizer_state": self.__dict__, "words": words,
                    "elapsed_seconds": elapsed_before + time.monotonic() - started})

        try:
            while len(self.vocab) < vocab_size:
                pair_counts = Counter()
                for symbols, frequency in words.items():
                    for pair in zip(symbols, symbols[1:]):
                        pair_counts[pair] += frequency
                if not pair_counts:
                    break
                pair = min(pair_counts, key=lambda p: (-pair_counts[p], self.names[p[0]], self.names[p[1]]))
                if pair_counts[pair] < 2:
                    break
                new_id = len(self.vocab)
                updated = Counter()
                for symbols, frequency in words.items():
                    result = []
                    j = 0
                    while j < len(symbols):
                        if j + 1 < len(symbols) and symbols[j:j + 2] == pair:
                            result.append(new_id)
                            j += 2
                        else:
                            result.append(symbols[j])
                            j += 1
                    updated[tuple(result)] += frequency
                # Commit the whole merge only after every word has been replaced.
                commit_in_progress = True
                self.vocab.append(self.vocab[pair[0]] + self.vocab[pair[1]])
                self.names.append(self.names[pair[0]] + self.names[pair[1]])
                self.merges[pair] = new_id
                self.vocab_size = len(self.vocab)
                words = updated
                commit_in_progress = False
                merges_this_call += 1
                if len(self.merges) % checkpoint_merges == 0 or time.monotonic() - last_saved >= 60:
                    snapshot()
                    print(f"BPE vocabulary {self.vocab_size:,}/{vocab_size:,} | "
                          f"elapsed {(elapsed_before + time.monotonic() - started)/60:.1f} min", flush=True)
                    last_saved = time.monotonic()
                if stop_after_merges is not None and merges_this_call >= stop_after_merges:
                    break
        finally:
            if not commit_in_progress:
                snapshot()
        return self

    def encode(self, text):
        result = []
        for piece in re.findall(r"\S+|\s+", text):
            symbols = [b + 4 for b in piece.encode("utf-8")] + [self.boundary_id]
            while len(symbols) > 1:
                candidates = [(self.merges[(a, b)], (a, b))
                              for a, b in zip(symbols, symbols[1:]) if (a, b) in self.merges]
                if not candidates:
                    break
                new_id, pair = min(candidates)
                merged = []
                j = 0
                while j < len(symbols):
                    if j + 1 < len(symbols) and (symbols[j], symbols[j + 1]) == pair:
                        merged.append(new_id)
                        j += 2
                    else:
                        merged.append(symbols[j])
                        j += 1
                symbols = merged
            result.extend(symbols)
        return result

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8")
''',
    'phase1_data.py': r'''"""Stream legal documents, freeze document ownership, and build token caches."""
import json
import os
import re
import time
from functools import lru_cache
from pathlib import Path
import ijson
import numpy as np
from phase1_utils import atomic_json, checked_json, read_json, sha256_file, text_hash

DTYPE = np.dtype("<u2")


def is_validation(digest):
    return int(digest[:16], 16) % 20 == 0


def iter_records(path):
    with open(path, "rb") as file:
        first = file.read(4096).lstrip()[:1]
        file.seek(0)
        if first == b"[":
            records = ijson.items(file, "item")
        elif first == b"{":
            records = (record for _, record in ijson.kvitems(file, ""))
        else:
            raise ValueError("Expected a JSON list or an ID-to-record dictionary")
        for record in records:
            if not isinstance(record, dict) or not isinstance(record.get("text"), str):
                raise ValueError("Every document must contain a string 'text'")
            yield record


def audit_source(path, source, folder, signature, seen,
                 bpe_chars=500_000, val_chars=500_000, generation_examples=25):
    """Exact duplicates belong to the first source in the frozen source order."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    sample_path, manifest_path = folder / f"{source}_samples.json", folder / f"{source}_ownership.jsonl"
    if sample_path.exists():
        saved = read_json(sample_path)
        if saved["signature"] != signature or sha256_file(manifest_path) != saved["ownership_sha256"]:
            raise ValueError("Saved source audit differs")
        with open(manifest_path, encoding="utf-8") as file:
            for line in file:
                digest = json.loads(line)["full_text_sha256"]
                if digest in seen:
                    raise ValueError("Cross-source ownership contains a duplicate")
                seen.add(digest)
        return saved
    temporary = manifest_path.with_suffix(".jsonl.tmp")
    bpe_chunks, val_texts, prompts, truths = [], [], {}, {}
    counts = {"records": 0, "empty": 0, "duplicates": 0, "owned_documents": 0,
              "owned_training_documents": 0, "owned_validation_documents": 0,
              "owned_training_characters": 0, "owned_validation_characters": 0,
              "bpe_characters": 0, "validation_sample_characters": 0}
    last_print = time.monotonic()
    with open(temporary, "w", encoding="utf-8") as manifest:
        for index, record in enumerate(iter_records(path)):
            counts["records"] += 1
            text = record["text"]
            if not text.strip():
                counts["empty"] += 1
                continue
            digest = text_hash(text)
            if digest in seen:
                counts["duplicates"] += 1
                continue
            seen.add(digest)
            validation = is_validation(digest)
            manifest.write(json.dumps({"record_index": index, "full_text_sha256": digest,
                                       "validation": validation, "characters": len(text)}) + "\n")
            counts["owned_documents"] += 1
            key = "validation" if validation else "training"
            counts[f"owned_{key}_documents"] += 1
            counts[f"owned_{key}_characters"] += len(text)
            if not validation and counts["bpe_characters"] < bpe_chars:
                chunk = text[:min(4000, bpe_chars - counts["bpe_characters"])]
                bpe_chunks.append(chunk)
                counts["bpe_characters"] += len(chunk)
            if validation:
                if counts["validation_sample_characters"] < val_chars:
                    chunk = text[:min(20_000, val_chars - counts["validation_sample_characters"])]
                    val_texts.append(chunk)
                    counts["validation_sample_characters"] += len(chunk)
                if len(prompts) < generation_examples:
                    words = text.split(maxsplit=640)
                    if len(words) >= 640:
                        prompt_id = f"{source}_{len(prompts):04d}"
                        prompts[prompt_id] = {"text": " ".join(words[:512])}
                        truths[prompt_id] = {"text": " ".join(words[512:640])}
            if time.monotonic() - last_print >= 30:
                print(source, "| scanned", counts["records"], "| duplicates", counts["duplicates"], flush=True)
                last_print = time.monotonic()
        manifest.flush()
        os.fsync(manifest.fileno())
    os.replace(temporary, manifest_path)
    if counts["bpe_characters"] != bpe_chars or counts["validation_sample_characters"] != val_chars:
        raise ValueError(f"Sample quotas not met for {source}: {counts}")
    if len(prompts) != generation_examples:
        raise ValueError(f"Only {len(prompts)} suitable held-out prompts in {source}")
    saved = {"signature": signature, "statistics": counts, "bpe_chunks": bpe_chunks,
             "validation_texts": val_texts, "prompts": prompts, "truths": truths,
             "ownership_sha256": sha256_file(manifest_path)}
    atomic_json(sample_path, saved)
    return saved


def cached_encoder(tokenizer):
    @lru_cache(maxsize=200_000)
    def piece_ids(piece):
        return tuple(tokenizer.encode(piece))
    def encode(text):
        return [i for piece in re.findall(r"\S+|\s+", text) for i in piece_ids(piece)]
    return encode


def cache_metadata(binary, metadata_path, signature):
    if not Path(metadata_path).exists():
        return None
    metadata = read_json(metadata_path)
    if metadata["signature"] != signature:
        raise ValueError("Token-cache signature differs")
    if Path(binary).stat().st_size != metadata["token_count"] * DTYPE.itemsize:
        raise ValueError("Token-cache size differs")
    if sha256_file(binary) != metadata["sha256"]:
        raise ValueError("Token-cache checksum differs")
    return metadata


def build_source_cache(path, ownership_path, encode, folder, signature, target_tokens):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    binary, metadata_path = folder / "train.bin", folder / "train_metadata.json"
    old = cache_metadata(binary, metadata_path, signature)
    if old is not None:
        return old
    owners = {}
    with open(ownership_path, encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)
            if not row["validation"]:
                owners[row["record_index"]] = row["full_text_sha256"]
    partial, progress_path = folder / "train.partial.bin", folder / "cache_progress.json"
    state = {"tokens_written": 0, "next_record_index": 0, "documents_encoded": 0,
             "last_document_truncated": False, "complete": False,
             "source_exhausted": False}
    if progress_path.exists():
        progress = read_json(progress_path)
        if progress["signature"] != signature:
            raise ValueError("Partial-cache signature differs")
        state = progress["state"]
    elif partial.exists():
        raise ValueError("Partial cache exists without progress metadata")
    if state["tokens_written"] and not partial.exists() and not binary.exists():
        raise ValueError("The partial cache needed for resumption is missing")
    if not state["complete"]:
        last_print = time.monotonic()
        with open(partial, "r+b" if partial.exists() else "w+b") as output:
            output.seek(state["tokens_written"] * DTYPE.itemsize)
            output.truncate()
            def snapshot():
                output.seek(state["tokens_written"] * DTYPE.itemsize)
                output.truncate()
                output.flush()
                os.fsync(output.fileno())
                atomic_json(progress_path, {"signature": signature, "state": state})
            try:
                for index, record in enumerate(iter_records(path)):
                    if index < state["next_record_index"]:
                        continue
                    if index in owners:
                        if text_hash(record["text"]) != owners[index]:
                            raise ValueError("Document content differs from the frozen audit")
                        ids = encode(record["text"]) + [3]
                        keep = ids[:target_tokens - state["tokens_written"]]
                        np.asarray(keep, dtype=DTYPE).tofile(output)
                        updated = dict(state)
                        updated["tokens_written"] += len(keep)
                        updated["documents_encoded"] += 1
                        updated["last_document_truncated"] = len(keep) < len(ids)
                        updated["next_record_index"] = index + 1
                        state = updated
                    else:
                        state["next_record_index"] = index + 1
                    if time.monotonic() - last_print >= 30:
                        snapshot()
                        print(signature["source"], "| cached", state["tokens_written"], "/", target_tokens, flush=True)
                        last_print = time.monotonic()
                    if state["tokens_written"] >= target_tokens:
                        state["complete"] = True
                        break
                else:
                    state["source_exhausted"] = True
                    state["complete"] = True
            finally:
                snapshot()
    source = partial if partial.exists() else binary
    if not source.exists() or state["tokens_written"] == 0:
        raise ValueError("No training tokens in this source")
    checksum = sha256_file(source)
    if partial.exists():
        os.replace(partial, binary)
    metadata = {"signature": signature, "dtype": DTYPE.str,
                "token_count": state["tokens_written"], "sha256": checksum,
                "requested_token_count": target_tokens,
                "source_exhausted": state["source_exhausted"], "statistics": state}
    atomic_json(metadata_path, metadata)
    return metadata


def build_validation_cache(texts, encode, folder, signature):
    folder = Path(folder)
    binary, metadata_path = folder / "validation.bin", folder / "validation_metadata.json"
    old = cache_metadata(binary, metadata_path, signature)
    if old is not None:
        return old
    temporary = binary.with_suffix(".bin.tmp")
    count = 0
    with open(temporary, "wb") as output:
        for text in texts:
            ids = encode(text) + [3]
            np.asarray(ids, dtype=DTYPE).tofile(output)
            count += len(ids)
    os.replace(temporary, binary)
    metadata = {"signature": signature, "dtype": DTYPE.str,
                "token_count": count, "sha256": sha256_file(binary), "documents": len(texts)}
    atomic_json(metadata_path, metadata)
    return metadata


def block_order(n_blocks, seed, source_number, epoch):
    return np.random.default_rng(seed + 100_000 * source_number + epoch).permutation(n_blocks)


def initial_cursors(sources):
    return {source: {"epoch": 0, "cursor": 0} for source in sources}


def mixed_batch(caches, cursors, source_order, source_offset, batch_size, block_size, seed, orders=None):
    xs, ys, counts = [], [], {source: 0 for source in source_order}
    for j in range(batch_size):
        source_number = (source_offset + j) % len(source_order)
        source = source_order[source_number]
        tokens = caches[source]
        n_blocks = (len(tokens) - 1) // block_size
        state = cursors[source]
        if state["cursor"] == n_blocks:
            state["epoch"] += 1
            state["cursor"] = 0
            if orders is not None:
                orders[source] = block_order(n_blocks, seed, source_number, state["epoch"])
        order = (block_order(n_blocks, seed, source_number, state["epoch"])
                 if orders is None else orders[source])
        index = int(order[state["cursor"]])
        state["cursor"] += 1
        start = index * block_size
        xs.append(np.asarray(tokens[start:start + block_size], dtype=np.int64))
        ys.append(np.asarray(tokens[start + 1:start + block_size + 1], dtype=np.int64))
        counts[source] += block_size
    return np.stack(xs), np.stack(ys), (source_offset + batch_size) % len(source_order), counts
''',
}
SRC_DIR = OUTPUT_ROOT / '_source'
SRC_DIR.mkdir(parents=True, exist_ok=True)
for name, source in modules.items():
    path = SRC_DIR / name
    if not path.exists() or path.read_bytes() != source.encode('utf-8'):
        path.write_bytes(source.encode('utf-8'))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [4]:
from phase1_utils import atomic_json, atomic_pickle, checked_json, make_bundle, read_json, sha256_file, text_hash, write_source
from a1_tokenizer import A1BPETokenizer

def resolve_input(explicit, names, marker, setting, *, required=True):
    """Prefer an explicit folder or a matching folder in the work area."""
    if explicit is not None:
        folder = Path(explicit).expanduser().resolve()
        if not (folder / marker).is_file():
            raise FileNotFoundError(f"{setting} must point to the folder containing {marker}: {folder}")
        return folder
    for root in (WORK_ROOT / "cs781_phase1", INPUT_ROOT):
        matches = sorted({
            path.parent.resolve()
            for path in root.rglob(marker)
            if path.parent.name in names and path.is_file()
        })
        if len(matches) > 1:
            paths = "\n".join(map(str, matches))
            raise ValueError(f"Set {setting} to one of these folders:\n{paths}")
        if matches:
            return matches[0]
    if required:
        raise FileNotFoundError(
            f"Attach an extracted {', '.join(names)} folder containing {marker}, or set {setting}."
        )
    return None


def restore_folder(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=destination.parent) as temporary:
        staged = Path(temporary) / "backup"
        shutil.copytree(source, staged)
        staged.rename(destination)


def show_download(path):
    display(FileLink(os.path.relpath(path, Path.cwd())))

def load_validation(prep_dir, sources):
    prompts = read_json(prep_dir / "validation_prompts.json")
    truths = read_json(prep_dir / "validation_truth.json")
    groups = read_json(prep_dir / "validation_groups.json")
    anchor = groups["anchor_50"]
    balanced = groups["balanced_100"]
    per_source = groups["balanced_source_ids"]
    if (
        len(prompts) != 125 or set(prompts) != set(truths)
        or len(anchor) != 50 or len(set(anchor)) != 50
        or len(balanced) != 100 or len(set(balanced)) != 100
        or set(per_source) != set(sources)
        or any(len(per_source[source]) != 25 for source in sources)
        or [key for source in sources for key in per_source[source]] != balanced
        or per_source["supreme_court"] != anchor[:25]
        or set(anchor) | set(balanced) != set(prompts)
    ):
        raise ValueError("The saved anchor, balanced, and per-source groups do not match the Phase 1 split.")
    for key in prompts:
        if len(prompts[key]["text"].split()) != 512 or len(truths[key]["text"].split()) != 128:
            raise ValueError(f"Unexpected prompt/reference length for {key}.")
    return prompts, truths, groups


def load_test_prompts(prep_dir):
    prompts = read_json(prep_dir / "official_test_prompts.json")
    if not isinstance(prompts, dict) or len(prompts) != 1000:
        raise ValueError("Expected 1,000 official prompt IDs.")
    if any(not isinstance(row, dict) or not isinstance(row.get("text"), str) or not row["text"].strip()
        for row in prompts.values()):
        raise ValueError("The official prompt file contains an empty or invalid text field.")
    return prompts


def check_hash(path, expected):
    if sha256_file(path) != expected:
        raise ValueError(f"SHA-256 mismatch: {path}")


def anchor_json_sha(records):
    # Same UTF-8 JSON representation used to write the prep_001 anchor files.
    return text_hash(json.dumps(records, indent=2, ensure_ascii=False))


## 3. Pin the corpus and resolve inputs

Record the exact corpus revision. A rebuild can use the original anchor files if supplied.
Otherwise it selects the first 50 eligible unique held-out Supreme Court documents within the
first 20,000 source records, using the original rule. Each example splits the first 640 words
into a 512-word prompt and a 128-word reference.


In [5]:
if REBUILD:
    from phase1_data import audit_source, build_source_cache, build_validation_cache, cached_encoder, is_validation, iter_records

    PREP_DIR = WORK_ROOT / "cs781_phase1" / "preparation" / PREPARATION_ID
    if PREP_INPUT is not None and Path(PREP_INPUT).expanduser().resolve() != PREP_DIR.resolve():
        partial = resolve_input(PREP_INPUT, (PREPARATION_ID,), "preparation_config.json", "PREP_INPUT")
        if not PREP_DIR.exists():
            print(f"Copying partial preparation to {PREP_DIR}", flush=True)
            restore_folder(partial, PREP_DIR)
        else:
            print(f"Using existing working preparation · {PREP_DIR}")
    PREP_DIR.mkdir(parents=True, exist_ok=True)
    for name in modules:
        write_source(PREP_DIR / "sources" / name, (SRC_DIR / name).read_bytes())

    REMOVE_DOWNLOADED_RAW_AFTER_STAGE = True
else:
    PREP_DIR = resolve_input(PREP_INPUT, (PREPARATION_ID,), "cache_manifest.json", "PREP_INPUT")

AUDIT_DIR = PREP_DIR / "audits"
config_path = PREP_DIR / "preparation_config.json"
raw_checksums_path = PREP_DIR / "raw_checksums.json"
print(f"Preparation folder · {PREP_DIR}", flush=True)


Preparation folder · /kaggle/input/datasets/ash9201kk/prep-2/cs781_phase1/preparation/prep_002_mixed16k


In [6]:
if REBUILD:
    print("Resolving pinned corpus files", flush=True)
    from huggingface_hub import HfApi, hf_hub_download
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN")
    api = HfApi(token=HF_TOKEN)
    info = api.dataset_info(HF_DATASET_REPO, revision=DATASET_REVISION, files_metadata=True)
    assert info.sha == DATASET_REVISION
    repo_files = {row.rfilename: row for row in info.siblings}
    def resolve_file(basename, split):
        expected = f'Phase_1_Pretraining/{split}/{basename}'
        if expected in repo_files:
            return expected
        matches = [name for name in repo_files if Path(name).name.lower() == basename.lower()
                   and 'phase_1' in name.lower() and f'/{split.lower()}/' in name.lower()]
        if len(matches) != 1:
            raise ValueError(f'Cannot uniquely locate {basename}: {matches}')
        return matches[0]
    FILENAMES = {source: resolve_file(f'train_{source}.json', 'Train') for source in SOURCE_ORDER}
    TEST_FILENAME = resolve_file('test.json', 'Test')
    RAW_DIR = WORK_ROOT / 'cs781_phase1_raw_downloads'
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    raw_checksums_path = PREP_DIR / 'raw_checksums.json'
    raw_checksums = read_json(raw_checksums_path) if raw_checksums_path.exists() else {}

    def get_raw(source):
        filename = FILENAMES[source]
        input_root = Path(RAW_INPUT_ROOT) if RAW_INPUT_ROOT else INPUT_ROOT
        input_matches = sorted(input_root.rglob(Path(filename).name))
        if len(input_matches) > 1:
            raise ValueError(f'Several raw input copies found. Set RAW_INPUT_ROOT to one dataset folder: {input_matches}')
        if input_matches:
            path = input_matches[0]
            checksum = sha256_file(path)
            lfs = getattr(repo_files[filename], 'lfs', None)
            expected_sha = lfs.get('sha256') if isinstance(lfs, dict) else getattr(lfs, 'sha256', None)
            expected_sha = expected_sha or raw_checksums.get(source)
            if not expected_sha or checksum != expected_sha:
                raise ValueError(f'Cannot verify raw input {path} against the pinned official source')
            raw_checksums[source] = checksum
            atomic_json(raw_checksums_path, raw_checksums)
            print('Using verified official raw input:', path)
            return path
        destination = RAW_DIR / filename
        size = repo_files[filename].size
        extra_needed = max(0, (size or 0) - (destination.stat().st_size if destination.exists() else 0))
        if extra_needed + 2 * 2**30 > shutil.disk_usage(WORK_ROOT).free:
            raise RuntimeError(f'Insufficient working disk for {source}. Attach this official raw source as input or free old output copies; keep backups.')
        path = Path(hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type='dataset',
            revision=DATASET_REVISION, token=HF_TOKEN, local_dir=RAW_DIR))
        checksum = sha256_file(path)
        if source in raw_checksums and raw_checksums[source] != checksum:
            raise ValueError('Pinned raw file checksum changed')
        raw_checksums[source] = checksum
        atomic_json(raw_checksums_path, raw_checksums)
        return path

    def release_raw(path):
        if REMOVE_DOWNLOADED_RAW_AFTER_STAGE:
            path = Path(path).resolve()
            if path.is_relative_to(RAW_DIR.resolve()):
                path.unlink(missing_ok=True)  # never delete a read-only input dataset

    print('Official sources:', FILENAMES)
    print('Working disk free GiB:', round(shutil.disk_usage(WORK_ROOT).free / 2**30, 2))
    print('Raw file GiB:', {s: round((repo_files[f].size or 0) / 2**30, 3) for s, f in FILENAMES.items()})
if REBUILD:
    if ANCHOR_INPUT is not None:
        anchor_dir = resolve_input(ANCHOR_INPUT, ("prep_001",), "preparation_config.json", "ANCHOR_INPUT")
        if read_json(anchor_dir / "preparation_config.json")["dataset_revision"] != DATASET_REVISION:
            raise ValueError("The supplied prep_001 anchors come from a different corpus revision.")
        anchor_prompt_path = anchor_dir / "validation_prompts.json"
        anchor_truth_path = anchor_dir / "validation_truth.json"
        anchor_prompts = read_json(anchor_prompt_path)
        anchor_truths = read_json(anchor_truth_path)
        anchor_records = read_json(anchor_dir / "sample_manifest.json")["records"]
        anchor_prompts_sha256 = sha256_file(anchor_prompt_path)
        anchor_truths_sha256 = sha256_file(anchor_truth_path)
        print("Loaded the original prep_001 anchors")
    else:
        anchor_cache = AUDIT_DIR / "supreme_court_anchor_50.json"
        if anchor_cache.is_file():
            anchors = read_json(anchor_cache)
            if (
                anchors["dataset_revision"] != DATASET_REVISION
                or anchors["raw_sha256"] != raw_checksums["supreme_court"]
            ):
                raise ValueError("The saved Supreme Court anchors come from different raw data.")
            print("Loaded the recovered Supreme Court anchors")
        else:
            print("Recovering the 50 Supreme Court anchors from the pinned corpus", flush=True)
            supreme_court_path = get_raw("supreme_court")
            anchor_prompts, anchor_truths, anchor_records = {}, {}, []
            seen_hashes = set()
            for index, record in enumerate(iter_records(supreme_court_path)):
                if index >= 20_000:
                    break
                text = record["text"]
                if not text.strip():
                    continue
                digest = text_hash(text)
                if digest in seen_hashes:
                    continue
                seen_hashes.add(digest)
                if not is_validation(digest):
                    continue
                words = text.split(maxsplit=640)
                if len(words) < 640:
                    continue
                key = f"val_{len(anchor_prompts):04d}"
                anchor_prompts[key] = {"text": " ".join(words[:512])}
                anchor_truths[key] = {"text": " ".join(words[512:640])}
                anchor_records.append({"generation_example_id": key, "full_text_sha256": digest})
                if len(anchor_prompts) == 50:
                    break
            del seen_hashes
            if len(anchor_prompts) != 50:
                raise ValueError("The pinned source did not yield all 50 Supreme Court anchors.")
            anchors = {
                "dataset_revision": DATASET_REVISION,
                "raw_sha256": raw_checksums["supreme_court"],
                "prompts": anchor_prompts,
                "truths": anchor_truths,
                "records": anchor_records,
            }
            AUDIT_DIR.mkdir(parents=True, exist_ok=True)
            atomic_json(anchor_cache, anchors)
        anchor_prompts = anchors["prompts"]
        anchor_truths = anchors["truths"]
        anchor_records = anchors["records"]
        anchor_prompts_sha256 = anchor_json_sha(anchor_prompts)
        anchor_truths_sha256 = anchor_json_sha(anchor_truths)

    if len(anchor_prompts) != 50 or anchor_prompts.keys() != anchor_truths.keys():
        raise ValueError("Expected the same 50 anchor IDs in the prompts and references.")
    configuration = {
        'preparation_id': PREPARATION_ID, 'dataset_repo': HF_DATASET_REPO,
        'dataset_revision': DATASET_REVISION, 'filenames': FILENAMES, 'test_filename': TEST_FILENAME,
        'source_order': SOURCE_ORDER, 'vocab_size': VOCAB_SIZE, 'context': MAX_SEQ_LEN,
        'bpe_chars_per_source': BPE_CHARS_PER_SOURCE,
        'validation_chars_per_source': VALIDATION_CHARS_PER_SOURCE,
        'cache_tokens_per_source': CACHE_TOKENS_PER_SOURCE,
        'split_rule': 'int(SHA256(full text UTF8)[:16], 16) % 20 == 0',
        'duplicate_rule': 'Exact full-text duplicates belong to the first source in source_order',
        'bpe_prefix_chars_per_document': 4000, 'validation_prefix_chars_per_document': 20000,
        'generation_prompt_words': 512, 'generation_reference_words': 128,
        'anchor_prompts_sha256': anchor_prompts_sha256,
        'anchor_truths_sha256': anchor_truths_sha256,
        'source_checksums': {name: sha256_file(PREP_DIR / 'sources' / name)
                             for name in ['phase1_utils.py', 'a1_tokenizer.py', 'phase1_data.py']}}
    checked_json(PREP_DIR / 'preparation_config.json', configuration)

config = read_json(config_path)
if not REBUILD:
    raw_checksums = read_json(raw_checksums_path)
if (
    config["preparation_id"] != PREPARATION_ID
    or config["dataset_repo"] != HF_DATASET_REPO
    or config["source_order"] != SOURCE_ORDER
    or config["vocab_size"] != VOCAB_SIZE
    or config["context"] != MAX_SEQ_LEN
    or config["bpe_chars_per_source"] != BPE_CHARS_PER_SOURCE
    or config["validation_chars_per_source"] != VALIDATION_CHARS_PER_SOURCE
    or config["cache_tokens_per_source"] != CACHE_TOKENS_PER_SOURCE
):
    raise ValueError("The preparation configuration does not match the Phase 1 mixed 16K setup.")
for name, digest in config["source_checksums"].items():
    check_hash(SRC_DIR / name, digest)
print(f"Corpus revision · {config['dataset_revision']}")


Corpus revision · 89edba6a44d6bb9593fd13b6895b5cd3b06eb82e


## 4. Audit document ownership

Hash the full original text before taking any prefix or token sample. Empty records and
later exact duplicates are excluded. The hash split assigns whole documents to training
or validation, and the fixed source order gives duplicates a single owner. The per-source
ownership ledgers connect each cache back to the original corpus audit.


In [7]:
if REBUILD:
    audits, seen = {}, set()  # fresh on every rerun; completed audits repopulate this set
    AUDIT_DIR = PREP_DIR / 'audits'
    for source in SOURCE_ORDER:
        sample_path = AUDIT_DIR / f'{source}_samples.json'
        if sample_path.exists():
            print(f"Restoring source audit · {source}", flush=True)
            raw_sha = raw_checksums[source]
            raw_path = RAW_DIR / FILENAMES[source]  # not read when the completed audit is reused
        else:
            print(f"Scanning raw source · {source}", flush=True)
            raw_path = get_raw(source)
            raw_sha = raw_checksums[source]
        signature = {'source': source, 'raw_sha256': raw_sha, 'configuration': configuration}
        audits[source] = audit_source(raw_path, source, AUDIT_DIR, signature, seen,
            BPE_CHARS_PER_SOURCE, VALIDATION_CHARS_PER_SOURCE, generation_examples=25)
        release_raw(raw_path)
    del seen
    gc.collect()
    atomic_json(PREP_DIR / 'audit_summary.json', {s: audits[s]['statistics'] for s in SOURCE_ORDER})

else:
    audits = {}
    for source in SOURCE_ORDER:
        print(f"Reading saved audit · {source}", flush=True)
        audit = read_json(AUDIT_DIR / f"{source}_samples.json")
        expected = {"source": source, "raw_sha256": raw_checksums[source], "configuration": config}
        if audit["signature"] != expected:
            raise ValueError(f"Audit signature differs for {source}.")
        check_hash(AUDIT_DIR / f"{source}_ownership.jsonl", audit["ownership_sha256"])
        audits[source] = audit

audit_summary = read_json(PREP_DIR / "audit_summary.json")
audit_rows = []
for source in SOURCE_ORDER:
    stats = audits[source]["statistics"]
    if stats != audit_summary[source]:
        raise ValueError(f"Saved audit counts differ for {source}.")
    audit_rows.append({
        "source": source,
        "raw_records": stats["records"],
        "exact_duplicates": stats["duplicates"],
        "training_documents": stats["owned_training_documents"],
        "held_out_documents": stats["owned_validation_documents"],
    })
display(pd.DataFrame(audit_rows))


Reading saved audit · supreme_court
Reading saved audit · high_courts
Reading saved audit · acts
Reading saved audit · tribunals


,source,raw_records,exact_duplicates,training_documents,held_out_documents
0,supreme_court,55549,165,52527,2836
1,high_courts,2536738,118437,2297010,121291
2,acts,18826,90,17797,935
3,tribunals,869132,4934,821107,43091


## 5. Assemble the generation validation sets

The balanced set takes 25 examples per source. Its Supreme Court cases are the first 25 of
the 50-case anchor set. Recover the sets in this order and check their IDs, texts, and word
counts. The merged file contains 125 unique held-out cases.


In [8]:
if REBUILD:
    remaining_hashes = {row["full_text_sha256"] for row in anchor_records if row.get("generation_example_id")}
    if len(remaining_hashes) != 50 or not all(is_validation(digest) for digest in remaining_hashes):
        raise ValueError("Expected 50 unique anchors from held-out documents.")
    with (AUDIT_DIR / "supreme_court_ownership.jsonl").open(encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)
            if row["validation"]:
                remaining_hashes.discard(row["full_text_sha256"])
            if not remaining_hashes:
                break
    if remaining_hashes:
        raise ValueError("Some anchors are absent from the held-out Supreme Court ownership ledger.")
else:
    saved_prompts, saved_truths, saved_groups = load_validation(PREP_DIR, SOURCE_ORDER)
    anchor_prompts = {key: saved_prompts[key] for key in saved_groups["anchor_50"]}
    anchor_truths = {key: saved_truths[key] for key in saved_groups["anchor_50"]}
    if (
        anchor_json_sha(anchor_prompts) != config["anchor_prompts_sha256"]
        or anchor_json_sha(anchor_truths) != config["anchor_truths_sha256"]
    ):
        raise ValueError("The saved anchor texts do not match the original anchor hashes.")

anchor_ids = list(anchor_prompts)
supreme = audits["supreme_court"]
if (
    list(supreme["prompts"].values()) != [anchor_prompts[key] for key in anchor_ids[:25]]
    or list(supreme["truths"].values()) != [anchor_truths[key] for key in anchor_ids[:25]]
):
    raise ValueError("The first 25 Supreme Court anchors differ from the source audit.")

prompts, truths = dict(anchor_prompts), dict(anchor_truths)
source_ids = {"supreme_court": anchor_ids[:25]}
for source in SOURCE_ORDER[1:]:
    prompts.update(audits[source]["prompts"])
    truths.update(audits[source]["truths"])
    source_ids[source] = list(audits[source]["prompts"])
groups = {
    "anchor_50": anchor_ids,
    "balanced_100": [key for source in SOURCE_ORDER for key in source_ids[source]],
    "balanced_source_ids": source_ids,
}
if REBUILD:
    checked_json(PREP_DIR / "validation_prompts.json", prompts)
    checked_json(PREP_DIR / "validation_truth.json", truths)
    checked_json(PREP_DIR / "validation_groups.json", groups)
    load_validation(PREP_DIR, SOURCE_ORDER)
elif (prompts, truths, groups) != (saved_prompts, saved_truths, saved_groups):
    raise ValueError("Reassembled validation groups differ from the saved preparation.")

print(f"Validation · {len(prompts)} unique cases · {len(groups['balanced_100'])} balanced · {len(anchor_ids)} anchor")
display(pd.DataFrame([{"source": source, "balanced_cases": len(source_ids[source])} for source in SOURCE_ORDER]))


Validation · 125 unique cases · 100 balanced · 50 anchor


,source,balanced_cases
0,supreme_court,25
1,high_courts,25
2,acts,25
3,tribunals,25


## 6. Fit or load the byte-BPE tokenizer

Sample up to 4,000 characters from each eligible training document until each source reaches
500,000 characters. Join the samples in source order. The tokenizer starts with four reserved
IDs, 256 byte symbols, and a piece-boundary symbol. BPE merges the most frequent adjacent pair
with the original deterministic tie-break. The 16,000-ID vocabulary contains 15,739 learned
merges. Rebuild mode can resume fitting from its merge checkpoint.

Recreate the fitting text from the audit samples and check its hash. In reuse mode, the
fit time comes from the saved fit summary. The text reconstruction and tokenizer round-trip
checks run in this session.


In [9]:
bpe_chunks = []
for source in SOURCE_ORDER:
    chunks = audits[source]["bpe_chunks"]
    if sum(map(len, chunks)) != BPE_CHARS_PER_SOURCE:
        raise ValueError(f"Unexpected BPE character sample size for {source}.")
    bpe_chunks.extend(chunks)
bpe_text = "\n\n".join(bpe_chunks)
bpe_text_path = PREP_DIR / "bpe_training_text.txt"
state_path = PREP_DIR / "tokenizer_state.pkl"
fit_path = PREP_DIR / "tokenizer_fitting_summary.json"
tokenizer_signature = {
    "vocab_size": VOCAB_SIZE,
    "training_text_sha256": text_hash(bpe_text),
    "algorithm_source_sha256": sha256_file(SRC_DIR / "a1_tokenizer.py"),
    "preparation_configuration_sha256": sha256_file(config_path),
}

if REBUILD:
    write_source(bpe_text_path, bpe_text)
else:
    check_hash(bpe_text_path, tokenizer_signature["training_text_sha256"])
    saved_manifest = read_json(PREP_DIR / "cache_manifest.json")
    if saved_manifest["tokenizer_signature"] != tokenizer_signature:
        raise ValueError("The reconstructed BPE sample or configuration differs from the cache manifest.")
    check_hash(state_path, saved_manifest["tokenizer_state_sha256"])

fit_now = REBUILD and not state_path.exists()
tokenizer = A1BPETokenizer()
if fit_now:
    print(f"Fitting {VOCAB_SIZE:,}-ID byte BPE", flush=True)
    started = time.monotonic()
    tokenizer.fit(bpe_text, VOCAB_SIZE, checkpoint_path=PREP_DIR / "bpe_fitting_checkpoint.pkl", checkpoint_merges=100)
    fit_elapsed = time.monotonic() - started
else:
    # Read tokenizer_state.pkl only from a trusted preparation backup.
    with state_path.open("rb") as file:
        saved = pickle.load(file)
    if saved["signature"] != tokenizer_signature:
        raise ValueError("Saved tokenizer signature differs from the reconstructed preparation.")
    tokenizer.__dict__.update(saved["state"])

if tokenizer.vocab_size != VOCAB_SIZE or len(tokenizer.merges) != VOCAB_SIZE - 261:
    raise ValueError("Unexpected vocabulary size or number of BPE merges.")
probes = ("The court allowed the appeal.\n", "हिन्दी café 🧪\t", "")
for sample in probes:
    if tokenizer.decode(tokenizer.encode(sample)) != sample:
        raise ValueError("Tokenizer round-trip check failed.")

fit_counts = {
    "vocabulary": tokenizer.vocab_size,
    "merged_pairs": len(tokenizer.merges),
    "training_characters_excluding_separators": sum(map(len, bpe_chunks)),
    "training_characters_including_separators": len(bpe_text),
}
if fit_now:
    fit = {**fit_counts, "elapsed_seconds_this_invocation": fit_elapsed}
    atomic_json(fit_path, fit)
    atomic_pickle(state_path, {"signature": tokenizer_signature, "state": tokenizer.__dict__})
else:
    fit = read_json(fit_path)
    if any(fit[key] != value for key, value in fit_counts.items()):
        raise ValueError("The saved fit summary differs from the tokenizer or fitting sample.")

if REBUILD:
    encode = cached_encoder(tokenizer)
    for sample in (*probes, bpe_text[:10_000]):
        if encode(sample) != tokenizer.encode(sample):
            raise ValueError("The cached encoder differs from direct BPE encoding.")

fit_origin = "this fitting invocation" if fit_now else "saved fitting invocation"
print(f"Tokenizer · {tokenizer.vocab_size:,} IDs · {len(tokenizer.merges):,} merges")
print(f"Fit time · {fit['elapsed_seconds_this_invocation'] / 60:.1f} min ({fit_origin})")
print(f"Fitting text · {len(bpe_text):,} characters including separators")


Tokenizer · 16,000 IDs · 15,739 merges
Fit time · 27.8 min (saved fitting invocation)
Fitting text · 2,001,638 characters including separators


## 7. Build or read the training caches

Append an end-of-document token and write each source's earliest unique training documents
up to the 75M-token quota. Encode separate validation samples for token-level loss evaluation.
The manifest records the tokenizer identity, source provenance, token counts and binary hashes.
The training notebooks sample full 2,560-token blocks from the four caches.


In [10]:
if REBUILD:
    cache_sources = {}
    for source in SOURCE_ORDER:
        print(f'Building or restoring token caches · {source}', flush=True)
        folder = PREP_DIR / 'caches' / source
        owner_path = AUDIT_DIR / f'{source}_ownership.jsonl'
        signature = {'source': source, 'tokenizer_signature': tokenizer_signature,
                     'ownership_sha256': sha256_file(owner_path), 'raw_sha256': raw_checksums[source],
                     'requested_training_tokens': CACHE_TOKENS_PER_SOURCE}
        raw_path = RAW_DIR / FILENAMES[source]
        if not (folder / 'train_metadata.json').exists():
            raw_path = get_raw(source)
        training_meta = build_source_cache(raw_path, owner_path, encode, folder, signature, CACHE_TOKENS_PER_SOURCE)
        release_raw(raw_path)
        val_signature = {'source': source, 'tokenizer_signature': tokenizer_signature,
            'validation_text_sha256': text_hash(json.dumps(audits[source]['validation_texts'], ensure_ascii=False))}
        validation_meta = build_validation_cache(audits[source]['validation_texts'], encode, folder, val_signature)
        assert training_meta['token_count'] >= MAX_SEQ_LEN + 1
        assert validation_meta['token_count'] >= MAX_SEQ_LEN + 1
        cache_sources[source] = {'training': training_meta, 'validation': validation_meta}
    manifest = {'preparation_id': PREPARATION_ID, 'source_order': SOURCE_ORDER,
                'tokenizer_signature': tokenizer_signature, 'tokenizer_state_sha256': sha256_file(state_path),
                'dtype': '<u2', 'sources': cache_sources,
                'mixture': {source: 0.25 for source in SOURCE_ORDER},
                'selection': 'Earliest unique training documents in each file, capped in token space'}
    checked_json(PREP_DIR / 'cache_manifest.json', manifest)

manifest = read_json(PREP_DIR / "cache_manifest.json")
if (
    manifest["preparation_id"] != PREPARATION_ID
    or manifest["source_order"] != SOURCE_ORDER
    or manifest["tokenizer_signature"] != tokenizer_signature
    or manifest["dtype"] != "<u2"
    or manifest["mixture"] != {source: 0.25 for source in SOURCE_ORDER}
):
    raise ValueError("The cache manifest differs from the mixed 16K setup.")

rows = []
for source, audit_row in zip(SOURCE_ORDER, audit_rows):
    print(f"Checking token caches · {source}", flush=True)
    counts = {}
    folder = PREP_DIR / "caches" / source
    for split, stem in (("training", "train"), ("validation", "validation")):
        metadata = read_json(folder / f"{stem}_metadata.json")
        binary = folder / f"{stem}.bin"
        expected_signature = {"source": source, "tokenizer_signature": tokenizer_signature}
        if split == "training":
            expected_signature.update(
                ownership_sha256=audits[source]["ownership_sha256"],
                raw_sha256=raw_checksums[source],
                requested_training_tokens=CACHE_TOKENS_PER_SOURCE,
            )
        else:
            text = json.dumps(audits[source]["validation_texts"], ensure_ascii=False)
            expected_signature["validation_text_sha256"] = text_hash(text)
        count = metadata["token_count"]
        if (
            metadata != manifest["sources"][source][split]
            or metadata["signature"] != expected_signature
            or count < MAX_SEQ_LEN + 1
            or metadata["dtype"] != "<u2"
            or binary.stat().st_size != count * 2
        ):
            raise ValueError(f"Cache metadata, provenance, or byte size differs: {binary}")
        if VERIFY_CACHE_HASHES:
            check_hash(binary, metadata["sha256"])
        counts[f"{split}_tokens"] = count
    rows.append({**audit_row, **counts})

total = sum(row["training_tokens"] for row in rows)
print(f"Prepared training caches · {total:,} tokens across {len(rows)} sources")
display(pd.DataFrame(rows)[["source", "training_tokens", "validation_tokens"]])


Checking token caches · supreme_court
Checking token caches · high_courts
Checking token caches · acts
Checking token caches · tribunals
Prepared training caches · 300,000,000 tokens across 4 sources


,source,training_tokens,validation_tokens
0,supreme_court,75000000,189364
1,high_courts,75000000,182098
2,acts,75000000,186021
3,tribunals,75000000,180485


## 8. Recompute prompt lengths and save diagnostics

Re-encode all held-out prompts, held-out prompts with references, and official test prompts.
Count tokens with the same byte-BPE tokenizer used by the models. The context and headroom
columns summarize prompts that could require truncation or leave limited room for generation.
The per-ID counts are compared with the original length audit in reuse mode.


In [11]:
if REBUILD:
    test_path = Path(hf_hub_download(
        repo_id=HF_DATASET_REPO, filename=TEST_FILENAME, repo_type="dataset",
        revision=DATASET_REVISION, token=HF_TOKEN, local_dir=RAW_DIR,
    ))
    checked_json(PREP_DIR / "official_test_prompts.json", read_json(test_path))
    release_raw(test_path)
test_prompts = load_test_prompts(PREP_DIR)


@lru_cache(maxsize=200_000)
def piece_length(piece):
    return len(tokenizer.encode(piece))


def count_tokens(text):
    # The tokenizer encodes whitespace and non-whitespace pieces independently.
    return sum(piece_length(piece) for piece in re.findall(r"\S+|\s+", text))


def measure_lengths(label, texts):
    print(f"Tokenizing {len(texts):,} {label}", flush=True)
    lengths = {}
    for index, (key, text) in enumerate(texts.items(), 1):
        lengths[key] = count_tokens(text)
        if index % 250 == 0 and index < len(texts):
            print(f"  {index:,}/{len(texts):,}", flush=True)
    return lengths


def summarize_lengths(lengths):
    values = list(lengths.values())
    return {
        "count": len(values), "min": min(values), "median": float(np.median(values)),
        "p95": float(np.percentile(values, 95)), "max": max(values),
        "over_2560": sum(value > MAX_SEQ_LEN for value in values),
        "less_than_512_tokens_headroom": sum(value > MAX_SEQ_LEN - 512 for value in values),
    }


for sample in (*probes, bpe_text[:10_000]):
    if count_tokens(sample) != len(tokenizer.encode(sample)):
        raise ValueError("The prompt-length counter differs from direct BPE encoding.")

length_groups = [
    ("heldout_prompt", "heldout_prompt_lengths", measure_lengths(
        "held-out prompts", {key: row["text"] for key, row in prompts.items()})),
    ("heldout_prompt_plus_reference", "heldout_total_lengths", measure_lengths(
        "held-out prompt/reference pairs", {key: row["text"] + " " + truths[key]["text"] for key, row in prompts.items()})),
    ("official_test_prompt", "official_test_prompt_lengths", measure_lengths(
        "official test prompts", {key: row["text"] for key, row in test_prompts.items()})),
]
length_rows = {name: summarize_lengths(lengths) for name, _, lengths in length_groups}
length_audit = {
    "context": MAX_SEQ_LEN, **length_rows,
    **{key: lengths for _, key, lengths in length_groups},
}
if REBUILD:
    checked_json(PREP_DIR / "length_audit.json", length_audit)
    atomic_json(PREP_DIR / "environment.json", {
        "python": platform.python_version(), "numpy": version("numpy"),
        "ijson": version("ijson"), "huggingface_hub": version("huggingface_hub"),
    })
else:
    recorded_lengths = read_json(PREP_DIR / "length_audit.json")
    if recorded_lengths["context"] != MAX_SEQ_LEN or any(
        recorded_lengths[key] != lengths for _, key, lengths in length_groups
    ):
        raise ValueError("The recomputed token lengths differ from the saved per-ID audit.")
    print("Recomputed token counts match the saved per-ID audit.")

display(pd.DataFrame.from_dict(length_rows, orient="index"))
length_path = OUTPUT_ROOT / "length_audit.json"
summary_path = OUTPUT_ROOT / "preparation_summary.json"
atomic_json(length_path, length_audit)
atomic_json(summary_path, {
    "mode": PREP_MODE,
    "preparation_id": PREPARATION_ID,
    "dataset_repo": config["dataset_repo"],
    "dataset_revision": config["dataset_revision"],
    "manifest_sha256": sha256_file(PREP_DIR / "cache_manifest.json"),
    "tokenizer_sha256": manifest["tokenizer_state_sha256"],
    "checked_binary_sha256": VERIFY_CACHE_HASHES,
    "document_audit": "built_or_resumed" if REBUILD else "loaded_saved",
    "tokenizer_fit": {"origin": "current_invocation" if fit_now else "saved", **fit},
    "token_caches": "built_or_resumed" if REBUILD else "loaded_saved",
    "training_tokens": total,
    "sources": rows,
    "generation_counts": {
        "anchor": len(anchor_ids), "balanced": len(groups["balanced_100"]),
        "heldout_unique": len(prompts), "test": len(test_prompts),
    },
    "prompt_lengths": {"origin": "recomputed", "groups": length_rows},
})
show_download(summary_path)
show_download(length_path)


Tokenizing 125 held-out prompts
Tokenizing 125 held-out prompt/reference pairs
Tokenizing 1,000 official test prompts
  250/1,000
  500/1,000
  750/1,000
Recomputed token counts match the saved per-ID audit.


,count,min,median,p95,max,over_2560,less_than_512_tokens_headroom
heldout_prompt,125,1041,1167.0,1300.8,1482,0,0
heldout_prompt_plus_reference,125,1297,1452.0,1598.2,1793,0,0
official_test_prompt,1000,1051,1134.0,1225.0,1635,0,0


/kaggle/working/phase1_preparation_outputs/preparation_summary.json

/kaggle/working/phase1_preparation_outputs/length_audit.json

## 9. Optional preparation archive

Set `CREATE_PREPARATION_ZIP = True` if you need a portable copy of the preparation.
Keep the archive and the source data in private storage. The model notebooks can read the
extracted folder, including when it is mounted as a Kaggle input.


In [12]:
if CREATE_PREPARATION_ZIP:
    backup_path = WORK_ROOT / f"{PREPARATION_ID}_backup.zip"
    required_bytes = sum(path.stat().st_size for path in PREP_DIR.rglob("*") if path.is_file())
    if shutil.disk_usage(WORK_ROOT).free < required_bytes + 512 * 2**20:
        raise RuntimeError("Working disk is too small for the archive. Save the preparation folder directly.")
    print("Writing preparation archive", flush=True)
    make_bundle(PREP_DIR, backup_path, PREP_DIR.parent)
    show_download(backup_path)


## Provenance

The corpus and generation format are from *CS781 Capstone*, Phase 1. The byte-BPE
implementation comes from the course assignments. Preparation and tokenizer hashes tie
this notebook's diagnostics to the artifacts used in model training.
